In [10]:
# use dataset 1 and Balanced Random Forest which be choosed by iM-seeker
import pandas as pd
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score, average_precision_score, precision_score, recall_score, f1_score, make_scorer
# from sklearn.model_selection import RandomizedSearchCV
import torch
import torch.nn as nn
import pickle
# from scipy.stats import randint
import warnings


# 加载数据
def load_data(file_positive, file_negative):
    # 读取正样本和负样本
    positive_samples = pd.read_csv(file_positive, sep='\t', header=None)
    negative_samples = pd.read_csv(file_negative, sep='\t', header=None)
    
    # 合并数据集
    data = pd.concat([positive_samples, negative_samples], axis=0)
    
    # 为正样本和负样本分配标签
    data['label'] = [1.] * len(positive_samples) + [0.] * len(negative_samples)
    
    # 分离特征和标签
    X = data.iloc[:, :-1]
    y = data['label']
    
    return X, y

# 训练集和测试集文件路径
train_positive_file = './33_feats/fold{}/positive_train_samples_dataset{}.csv'
train_negative_file = './33_feats/fold{}/negative_train_samples_dataset{}.csv'
test_positive_file = './33_feats/fold{}/positive_test_samples_dataset{}.csv'
test_negative_file = './33_feats/fold{}/negative_test_samples_dataset{}.csv'

In [ ]:
# 损失函数等价于 nn.CrossEntropyLoss()
# loss_scorer = make_scorer(log_loss, needs_proba=True, greater_is_better=False)

# 参数空间
# first time
# param_dist = {
#     'n_estimators': [64, 96, 128, 256],
#     'max_depth': [10, 15, 20, 25, 30, 40, None],
#     'max_features': ['sqrt', 'log2', None],
#     # 'min_samples_split': [2, 5, 10],
#     # 'min_samples_leaf': [1, 2, 4, 8],
#     # 'sampling_strategy': ['all', 'auto', 'not majority']
# }
# Best param (256, 20, 'sqrt')
# 2nd time
param_dist = {
    'n_estimators': [128, 256, 384, 512, 768, 1024],
    'max_depth': [10, 20, 30, 40, None],
    'max_features': ['sqrt'],
    # 'min_samples_split': [2, 5, 10],
    # 'min_samples_leaf': [1, 2, 4, 8],
    # 'sampling_strategy': ['all', 'auto', 'not majority']
}
# (1024, 20, 'sqrt') [0.25358161308583993, tensor(0.2773)]
# (512, 20, 'sqrt') [0.25369916725489866, tensor(0.2756)]

# 准备训练
best_params_dict = dict()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for n_est in param_dist['n_estimators']:
        for mdep in param_dist['max_depth']:
            for mfeat in param_dist['max_features']:
                best_params_dict[(n_est, mdep, mfeat)] = [0, 0]
                print(f"=== n_estimators: {n_est}, max_depth: {mdep}, max_features: {mfeat} ===")
                for fold in range(5):  # fold 是 0~4
                    for dataset_id in range(1,5):  # dataset 是 1~4
                        # print(f"\n=== Fold {fold}, Dataset {dataset_id} ===")
                        
                        # 构建路径
                        train_pos_path = f'./33_feats/fold{fold}/positive_train_samples_dataset{dataset_id}.csv'
                        train_neg_path = f'./33_feats/fold{fold}/negative_train_samples_dataset{dataset_id}.csv'
                        test_pos_path = f'./33_feats/fold{fold}/positive_test_samples_dataset{dataset_id}.csv'
                        test_neg_path = f'./33_feats/fold{fold}/negative_test_samples_dataset{dataset_id}.csv'

                        # 加载数据
                        X_train, y_train = load_data(train_pos_path, train_neg_path)
                        X_test, y_test = load_data(test_pos_path, test_neg_path)

                        # 初始化模型
                        brf = BalancedRandomForestClassifier(n_estimators=n_est, max_depth=mdep, max_features=mfeat, random_state=42)

                        brf.fit(X_train, y_train)

                        # 使用模型进行预测
                        y_pred = brf.predict(X_test)

                        # 计算交叉熵损失
                        cross_entropy_loss1 = log_loss(y_test, brf.predict_proba(X_test)[:, 1])
                        print(f'Cross-Entropy Loss1: {cross_entropy_loss1}')


                        y_pred_probs = brf.predict_proba(X_test)
                        y_pred_probs = torch.tensor(y_pred_probs, dtype=torch.float32)
                        y_pred_probs -= 0.5

                        # 真实标签也需要转换为Tensor格式
                        y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)
                        # 初始化交叉熵损失函数
                        criterion = nn.CrossEntropyLoss()

                        # 直接计算交叉熵损失
                        cross_entropy_loss2 = criterion(y_pred_probs[:2], y_test_tensor[:2])
                        print(f'Cross-Entropy Loss2: {cross_entropy_loss2.item()}')

                        best_params_dict[(n_est, mdep, mfeat)][0] += cross_entropy_loss1
                        best_params_dict[(n_est, mdep, mfeat)][1] += cross_entropy_loss2
                best_params_dict[(n_est, mdep, mfeat)][0] = best_params_dict[(n_est, mdep, mfeat)][0] / 36
                best_params_dict[(n_est, mdep, mfeat)][1] = best_params_dict[(n_est, mdep, mfeat)][1] / 36

print("Best parameters across all folds:", best_params_dict)

=== n_estimators: 128, max_depth: 10, max_features: sqrt ===
Cross-Entropy Loss1: 0.5059372769145786
Cross-Entropy Loss2: 0.6077010631561279
Cross-Entropy Loss1: 0.5081998799759618
Cross-Entropy Loss2: 0.5729895830154419
Cross-Entropy Loss1: 0.501758930348788
Cross-Entropy Loss2: 0.6760791540145874
Cross-Entropy Loss1: 0.5025524898806064
Cross-Entropy Loss2: 0.6536397337913513
Cross-Entropy Loss1: 0.4888526872938397
Cross-Entropy Loss2: 0.5346508026123047
Cross-Entropy Loss1: 0.48947507208189606
Cross-Entropy Loss2: 0.48315465450286865
Cross-Entropy Loss1: 0.49716254671358684
Cross-Entropy Loss2: 0.49188727140426636
Cross-Entropy Loss1: 0.4988474786817442
Cross-Entropy Loss2: 0.4134165048599243
Cross-Entropy Loss1: 0.4879248315625199
Cross-Entropy Loss2: 0.4205443263053894
Cross-Entropy Loss1: 0.487240997945799
Cross-Entropy Loss2: 0.4123210906982422
Cross-Entropy Loss1: 0.4892972428892179
Cross-Entropy Loss2: 0.3907702565193176
Cross-Entropy Loss1: 0.49092158697705734
Cross-Entropy Lo

In [26]:
min_loss = 100
best_params = {}
for k in best_params_dict:
    if min_loss is None or best_params_dict[k][0] < min_loss:
        min_loss = best_params_dict[k][0]
        best_params = k
print(best_params, best_params_dict[best_params])
min_loss = 100
best_params = {}
for k in best_params_dict:
    if min_loss is None or best_params_dict[k][1] < min_loss:
        min_loss = best_params_dict[k][1]
        best_params = k
print(best_params, best_params_dict[best_params])

(1024, 20, 'sqrt') [0.25358161308583993, tensor(0.2773)]
(512, 20, 'sqrt') [0.25369916725489866, tensor(0.2756)]


In [28]:
# 加载训练集和测试集
# X_train, y_train = load_data(train_positive_file, train_negative_file)
# X_test, y_test = load_data(test_positive_file, test_negative_file)

# 初始化 BalancedRandomForestClassifier
'''
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 
'criterion': 'gini', 'max_depth': 32, 'max_features': 'sqrt', 
'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 
'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 
'n_estimators': 200, 'n_jobs': None, 'oob_score': False, 'random_state': 0, 
'replacement': False, 'sampling_strategy': 'all', 'verbose': 0, 'warm_start': False}
'''
# brf = BalancedRandomForestClassifier(n_estimators=200, max_depth=32, max_features='sqrt', random_state=0) # params in iM-Seeker paper

metric_dict = {}
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for fold in range(5):  # fold 是 0~4
        print(f"=== fold: {fold} ===")
        metric_dict[fold] = [0]*7
        for dataset_id in range(1,5):  # dataset 是 1~4
            # print(f"\n=== Fold {fold}, Dataset {dataset_id} ===")
            
            # 构建路径
            train_pos_path = f'./33_feats/fold{fold}/positive_train_samples_dataset{dataset_id}.csv'
            train_neg_path = f'./33_feats/fold{fold}/negative_train_samples_dataset{dataset_id}.csv'
            test_pos_path = f'./33_feats/fold{fold}/positive_test_samples_dataset{dataset_id}.csv'
            test_neg_path = f'./33_feats/fold{fold}/negative_test_samples_dataset{dataset_id}.csv'

            # 加载数据
            X_train, y_train = load_data(train_pos_path, train_neg_path)
            X_test, y_test = load_data(test_pos_path, test_neg_path)

            # 初始化模型
            brf = BalancedRandomForestClassifier(n_estimators=512, max_depth=20, max_features='sqrt', random_state=42)

            brf.fit(X_train, y_train)

            # 使用模型进行预测
            y_pred = brf.predict(X_test)

            accuracy = accuracy_score(y_test, y_pred)
            print(f'Accuracy: {accuracy}')
            metric_dict[fold][0] += accuracy

            y_pred_probs = brf.predict_proba(X_test)
            y_pred_probs = torch.tensor(y_pred_probs, dtype=torch.float32)
            y_pred_probs -= 0.5

            # 真实标签也需要转换为Tensor格式
            y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)
            # 初始化交叉熵损失函数
            criterion = nn.CrossEntropyLoss()

            # 直接计算交叉熵损失
            cross_entropy_loss2 = criterion(y_pred_probs[:2], y_test_tensor[:2])
            print(f'Cross-Entropy Loss2: {cross_entropy_loss2.item()}')
            metric_dict[fold][1] += cross_entropy_loss2.item()

            # 计算AUROC
            auroc = roc_auc_score(y_test, brf.predict_proba(X_test)[:, 1])
            print(f'AUROC: {auroc}')
            metric_dict[fold][2] += auroc

            # 计算AUPRC
            auprc = average_precision_score(y_test, brf.predict_proba(X_test)[:, 1])
            print(f'AUPRC: {auprc}')
            metric_dict[fold][3] += auprc

            # 计算Precision, Recall, F1-score
            precision = precision_score(y_test, y_pred)
            recall = recall_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred)
            print(f'Precision: {precision}')
            print(f'Recall: {recall}')
            print(f'F1-score: {f1}')
            metric_dict[fold][4] += precision
            metric_dict[fold][5] += recall
            metric_dict[fold][6] += f1
        for i in range(7):
            metric_dict[fold][i] /= 4
print(metric_dict)


=== fold: 0 ===
Accuracy: 0.7869152046783626
Cross-Entropy Loss2: 0.6038522720336914
AUROC: 0.8798449063550883
AUPRC: 0.8777531905687228
Precision: 0.8744650499286734
Recall: 0.6709230207953302
F1-score: 0.759289843104872
Accuracy: 0.7918494152046783
Cross-Entropy Loss2: 0.5737309455871582
AUROC: 0.8794235012465816
AUPRC: 0.8745369746651208
Precision: 0.8817921830314586
Recall: 0.67493615468807
F1-score: 0.7646207894193016
Accuracy: 0.7876074498567335
Cross-Entropy Loss2: 0.6164260506629944
AUROC: 0.875834696082812
AUPRC: 0.8730184915327228
Precision: 0.8749423165666821
Recall: 0.6744930629669157
F1-score: 0.7617517075130574
Accuracy: 0.7845630372492837
Cross-Entropy Loss2: 0.6119890809059143
AUROC: 0.8749382128295888
AUPRC: 0.8737265148110377
Precision: 0.8691460055096418
Recall: 0.6734258271077909
F1-score: 0.7588695129284426
=== fold: 1 ===
Accuracy: 0.8043120774712224
Cross-Entropy Loss2: 0.5598487854003906
AUROC: 0.8962965485376088
AUPRC: 0.8918456294114715
Precision: 0.8843850760

In [29]:
train_pos_path = f'./33_feats/fold{fold}/positive_train_samples_dataset{dataset_id}.csv'
train_neg_path = f'./33_feats/fold{fold}/negative_train_samples_dataset{dataset_id}.csv'
test_pos_path = f'./33_feats/fold{fold}/positive_test_samples_dataset{dataset_id}.csv'
test_neg_path = f'./33_feats/fold{fold}/negative_test_samples_dataset{dataset_id}.csv'

# 加载数据
X_train, y_train = load_data(train_pos_path, train_neg_path)
X_test, y_test = load_data(test_pos_path, test_neg_path)

# 初始化模型
brf = BalancedRandomForestClassifier(n_estimators=512, max_depth=20, max_features='sqrt', random_state=42)

brf.fit(X_train, y_train)

# 使用模型进行预测
y_pred = brf.predict(X_test)

y_pred_probs = brf.predict_proba(X_test)
y_pred_probs = torch.tensor(y_pred_probs, dtype=torch.float32)
y_pred_probs -= 0.5

d:\miniconda3\envs\imotif\Lib\site-packages\imblearn\ensemble\_forest.py:577: FutureWarning: The default of `sampling_strategy` will change from `'auto'` to `'all'` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `'all'` to silence this warning and adopt the future behaviour.
  warn(
d:\miniconda3\envs\imotif\Lib\site-packages\imblearn\ensemble\_forest.py:589: FutureWarning: The default of `replacement` will change from `False` to `True` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `True` to silence this warning and adopt the future behaviour.
  warn(
d:\miniconda3\envs\imotif\Lib\site-packages\imblearn\ensemble\_forest.py:601: FutureWarning: The default of `bootstrap` will change from `True` to `False` in version 0.13. This change will follow the implementation proposed in the original paper. Set to `False` to silence this warning and adopt the future behaviour.
  warn(


In [30]:
# features importance
features33_names = [
    "[1] C-tract length",
    "[2] iM length",
    "[3] loop length",
    "[4] middle loop length",
    "[5] longest side loop length",
    "[6] shortest side loop length",
    "[7] sum of two side loops",
    "[8] longest loop length",
    "[9] shortest loop length",
    "[10] A density in iMs",
    "[11] C density in iMs",
    "[12] G density in iMs",
    "[13] T density in iMs",
    "[14] A density in loops",
    "[15] C density in loops",
    "[16] G density in loops",
    "[17] T density in loops",
    "[18] A density in middle loop", 
    "[19] C density in middle loop",
    "[20] G density in middle loop",
    "[21] T density in middle loop",
    "[22] A density in longest side loop",
    "[23] C density in longest side loop",
    "[24] G density in longest side loop",
    "[25] T density in longest side loop",
    "[26] A density in shortest side loop",
    "[27] C density in shortest side loop",
    "[28] G density in shortest side loop",
    "[29] T density in shortest side loop",
    "[30] A density in two side loops",
    "[31] C density in two side loops",
    "[32] G density in two side loops",
    "[33] T density in two side loops"
]

# importances = brf.feature_importances_
# indices = importances.argsort()[::-1]
# print('Feature ranking:')
# for f in range(X_train.shape[1]):
#     print(f'{f+1}. {features33_names[indices[f]]}: {importances[indices[f]]}')

# compute Pearson correlation coefficient between 33 features and prediction probabilities
from scipy.stats import pearsonr
correlations = list()
for i in range(X_train.shape[1]):
    corr, _ = pearsonr(X_test.iloc[:, i], y_pred_probs[:, 1])
    correlations.append([features33_names[i], corr])
correlations.sort(key=lambda x: x[1], reverse=True)
for i in range(len(correlations)):
    print(f"{i+1}th: {correlations[i]}")

1th: ['[16] G density in loops', 0.569713422267371]
2th: ['[32] G density in two side loops', 0.5663328768582272]
3th: ['[28] G density in shortest side loop', 0.5627000786891192]
4th: ['[12] G density in iMs', 0.4325643301312979]
5th: ['[24] G density in longest side loop', 0.4248257312194291]
6th: ['[20] G density in middle loop', 0.37882687031489204]
7th: ['[11] C density in iMs', 0.33526029099814164]
8th: ['[15] C density in loops', 0.16884137598675472]
9th: ['[19] C density in middle loop', 0.12725188921709943]
10th: ['[1] C-tract length', 0.11381689581996242]
11th: ['[23] C density in longest side loop', 0.059525433004079406]
12th: ['[31] C density in two side loops', 0.0541206599183783]
13th: ['[27] C density in shortest side loop', -0.039759912958292655]
14th: ['[4] middle loop length', -0.10406322335376844]
15th: ['[8] longest loop length', -0.14626070802318228]
16th: ['[2] iM length', -0.1884162398916488]
17th: ['[5] longest side loop length', -0.2106754645184006]
18th: ['[6]

In [27]:
## output to file
# with open("correlations.csv", "w") as f:
#     for i in range(33):
#         f.write(f"{i+1},{correlations[i][0]},{correlations[i][1]}\n")

In [31]:
from scipy.stats import spearmanr

correlations2 = list()
for i in range(33):
    rho, pval = spearmanr(X_test.iloc[:, i], y_pred_probs[:, 1])
    correlations2.append([features33_names[i], rho, pval])
correlations2.sort(key=lambda x: x[1], reverse=True)
for i in range(33):
    print(f"{i+1}th: {correlations2[i][0]}, Spearman rho = {correlations2[i][1]:.3f}, p = {correlations2[i][2]:.3e}")

1th: [16] G density in loops, Spearman rho = 0.591, p = 0.000e+00
2th: [32] G density in two side loops, Spearman rho = 0.561, p = 0.000e+00
3th: [28] G density in shortest side loop, Spearman rho = 0.511, p = 0.000e+00
4th: [12] G density in iMs, Spearman rho = 0.470, p = 1.831e-304
5th: [24] G density in longest side loop, Spearman rho = 0.412, p = 7.539e-227
6th: [20] G density in middle loop, Spearman rho = 0.400, p = 5.774e-213
7th: [11] C density in iMs, Spearman rho = 0.323, p = 3.740e-135
8th: [15] C density in loops, Spearman rho = 0.184, p = 1.535e-43
9th: [19] C density in middle loop, Spearman rho = 0.132, p = 6.466e-23
10th: [1] C-tract length, Spearman rho = 0.108, p = 5.468e-16
11th: [23] C density in longest side loop, Spearman rho = 0.073, p = 5.648e-08
12th: [31] C density in two side loops, Spearman rho = 0.070, p = 1.562e-07
13th: [27] C density in shortest side loop, Spearman rho = -0.079, p = 3.598e-09
14th: [4] middle loop length, Spearman rho = -0.104, p = 7.887

In [32]:
from sklearn.feature_selection import mutual_info_regression

correlations_mi1 = mutual_info_regression(X_test, y_pred_probs[:, 1])
correlations_mi = list()
for i in range(33):
    correlations_mi.append([features33_names[i], correlations_mi1[i]])
correlations_mi.sort(key=lambda x: x[1], reverse=True)
for i in range(33):
    print(f"{i+1}th: {correlations_mi[i][0]}, MI = {correlations_mi[i][1]:.3f}")


1th: [16] G density in loops, MI = 0.269
2th: [32] G density in two side loops, MI = 0.269
3th: [28] G density in shortest side loop, MI = 0.220
4th: [12] G density in iMs, MI = 0.202
5th: [13] T density in iMs, MI = 0.195
6th: [10] A density in iMs, MI = 0.189
7th: [33] T density in two side loops, MI = 0.176
8th: [17] T density in loops, MI = 0.172
9th: [11] C density in iMs, MI = 0.169
10th: [24] G density in longest side loop, MI = 0.169
11th: [14] A density in loops, MI = 0.163
12th: [30] A density in two side loops, MI = 0.148
13th: [20] G density in middle loop, MI = 0.142
14th: [3] loop length, MI = 0.126
15th: [25] T density in longest side loop, MI = 0.122
16th: [15] C density in loops, MI = 0.120
17th: [29] T density in shortest side loop, MI = 0.119
18th: [7] sum of two side loops, MI = 0.116
19th: [31] C density in two side loops, MI = 0.109
20th: [22] A density in longest side loop, MI = 0.106
21th: [5] longest side loop length, MI = 0.105
22th: [6] shortest side loop len

In [36]:
correlation_pearson_dict = {i[0]:i[1] for i in correlations}
correlation_spearman_dict = {i[0]:[i[1],i[2]] for i in correlations2}
correlation_mi_dict = {i[0]:i[1] for i in correlations_mi}

with open("./seeker_importance.csv", 'w') as f:
    # feature_name, spearman, p, mi, pearson
    f.write("feature_name,spearman,p,mi,pearson\n")
    for i in range(33):
        f.write(f"{features33_names[i]},{correlation_spearman_dict[features33_names[i]][0]},{correlation_spearman_dict[features33_names[i]][1]},{correlation_mi_dict[features33_names[i]]},{correlation_pearson_dict[features33_names[i]]}\n")